### Simple Attention Mechanism

In [12]:
import torch

inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [13]:
#Demonstrating attention mechanism for eg -journey x^2
input_query = inputs[1]
print(input_query)
len(inputs)

tensor([0.5500, 0.8700, 0.6600])


6

In [21]:
# Perform dot product with the rest of the vectors
atten_score = []
for i in range(len(inputs)):
    score = torch.dot(inputs[i],input_query)
    atten_score.append(score)
print(atten_score)


[tensor(0.9544), tensor(1.4950), tensor(1.4754), tensor(0.8434), tensor(0.7070), tensor(1.0865)]


In [30]:
inputs.shape

torch.Size([6, 3])

In [34]:
atten_score2 = torch.empty(inputs.shape[0])
for i in range(len(inputs)):
    atten_score2[i] = torch.dot(inputs[i],input_query)
print(atten_score2)

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])


In [35]:
#Normalisation simple
attn_weights_tmp = atten_score2/sum(atten_score2)
print(attn_weights_tmp)

tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656])


In [40]:
#Normalisation py torch version
attn_weights_norm2 = torch.softmax(atten_score2,dim=0)
print(attn_weights_norm2)

tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581])


In [57]:
#Multiply each normalised attention weight to the input vector to get the context vector 
context2 = torch.zeros(inputs.shape[1])
for idx,value in enumerate(inputs):
    context2 += attn_weights_norm2[idx]*inputs[idx]

print(context2)

tensor([0.4419, 0.6515, 0.5683])


In [48]:
input_query.shape

torch.Size([3])

In [68]:
#Performing for all using for loops
context_vect= []
for i,value in enumerate(inputs):
    input_query = inputs[i]
    atten_weight_temp = torch.zeros(inputs.shape[0])
    atten_weight = torch.zeros(inputs.shape[0])
    context_vect_ind = torch.zeros(input_query.shape)
    
    for j,value in enumerate(inputs):
        atten_weight_temp[j] = torch.dot(input_query,inputs[j])
    atten_weight = torch.softmax(atten_weight_temp,dim=0)
    
    for j,value in enumerate(inputs):
        context_vect_ind += atten_weight[j]*inputs[j]
    context_vect.append(context_vect_ind)
print(context_vect)        

[tensor([0.4421, 0.5931, 0.5790]), tensor([0.4419, 0.6515, 0.5683]), tensor([0.4431, 0.6496, 0.5671]), tensor([0.4304, 0.6298, 0.5510]), tensor([0.4671, 0.5910, 0.5266]), tensor([0.4177, 0.6503, 0.5645])]


In [73]:
# Using matrix multiplication
attn_score = inputs @ inputs.T
attn_weight = torch.softmax(attn_score,dim=1)
#1D, dim=0 = “use all elements”
#Important for matrix
#dim=0 → normalizes columns
#dim=1 → normalizes rows
all_context_vectors = attn_weight @ inputs

### Self-attention with trainable weights

In [74]:
# Performing the operation for x^2 as we did before
x2 = inputs[1]
d_in = inputs.shape[1] # dimension of input vector
d_out = 2 # the aim to generate a context vector of dimension 2

In [76]:
torch.manual_seed(123)

W_query = torch.nn.Parameter(torch.rand(d_in,d_out))
W_key = torch.nn.Parameter(torch.rand(d_in,d_out))
W_value = torch.nn.Parameter(torch.rand(d_in,d_out))

In [80]:
query_2 = x2 @ W_query
query_2

tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)

In [82]:
keys =  inputs @ W_key
values =  inputs @ W_value

In [87]:
keys_2 = keys[1]
attn_score22 = torch.dot(query_2,keys_2)
attn_weight = torch.softmax(attn_score,dim=0)

In [88]:
attn_score22

tensor(1.8524, grad_fn=<DotBackward0>)

In [94]:
attn_score2 = query_2 @ keys.T

In [104]:
d_k = keys.shape[1] #dimension of the input query
attn_weight2 = torch.softmax(attn_score2/ d_k**0.5,dim=-1)

In [105]:
attn_weight2

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
       grad_fn=<SoftmaxBackward0>)

In [106]:
context_vect2 = attn_weight2 @ values

In [107]:
context_vect2

tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)

### Expanding the code for all input vectors

In [108]:
import torch.nn as nn

class SelfAttention_v1(nn.Module):
    
    def __init__(self,d_in,d_out):
        super().__init__()
        self.W_query = torch.nn.Parameter(torch.rand(d_in,d_out))
        self.W_key = torch.nn.Parameter(torch.rand(d_in,d_out))
        self.W_value = torch.nn.Parameter(torch.rand(d_in,d_out))

    def forward(self,x):
        query = inputs @ self.W_query
        keys =  inputs @ self.W_key
        values =  inputs @ self.W_value

        attn_scores = query @ keys.T
        attn_weights = torch.softmax(attn_scores/ d_k**0.5,dim=-1)
        context_vects = attn_weights @ values

        return context_vects
        
torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in,d_out)
sa_v1(inputs)      
        

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)

### Using linear layer instead of using Parameter

In [152]:
import torch.nn as nn

class SelfAttention_v2(nn.Module):
    
    def __init__(self,d_in,d_out,qkv_bias = False):
        super().__init__()
        self.W_query = torch.nn.Linear(d_in,d_out,bias = qkv_bias)
        self.W_key = torch.nn.Linear(d_in,d_out,bias = qkv_bias)
        self.W_value = torch.nn.Linear(d_in,d_out,bias = qkv_bias)

    def forward(self,inputs):
        query = self.W_query(inputs)
        keys =  self.W_key(inputs)
        values =  self.W_value(inputs)

        attn_scores = query @ keys.T
        attn_weights = torch.softmax(attn_scores/ d_k**0.5,dim=-1)
        context_vects = attn_weights @ values

        return context_vects
        
torch.manual_seed(121)
sa_v2 = SelfAttention_v2(d_in,d_out)
sa_v2(inputs)      
        

tensor([[0.1988, 0.0211],
        [0.1985, 0.0211],
        [0.1987, 0.0210],
        [0.2023, 0.0178],
        [0.2029, 0.0171],
        [0.2009, 0.0191]], grad_fn=<MmBackward0>)

### Hiding future words with casual attention

In [115]:
#Casual attention is done to mask the future words. If the LLM sees the future words its data leakage, masking enforces the model to learn

query = sa_v2.W_query(inputs)
keys =  sa_v2.W_key(inputs)
values =  sa_v2.W_value(inputs)

attn_scores = query @ keys.T
attn_weights = torch.softmax(attn_scores/ d_k**0.5,dim=-1)


In [116]:
attn_weights

tensor([[0.1637, 0.1700, 0.1710, 0.1594, 0.1848, 0.1511],
        [0.1636, 0.1691, 0.1702, 0.1601, 0.1854, 0.1515],
        [0.1637, 0.1690, 0.1701, 0.1603, 0.1851, 0.1518],
        [0.1652, 0.1680, 0.1686, 0.1633, 0.1762, 0.1587],
        [0.1655, 0.1667, 0.1672, 0.1650, 0.1747, 0.1609],
        [0.1646, 0.1690, 0.1698, 0.1615, 0.1796, 0.1554]],
       grad_fn=<SoftmaxBackward0>)

In [124]:
context_length = attn_weights.shape[0]
mask_simple = torch.tril(torch.ones(context_length,context_length))
masked_simple = attn_weights * mask_simple # perform individual multiplication

In [125]:
masked_simple

tensor([[0.1637, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1636, 0.1691, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1637, 0.1690, 0.1701, 0.0000, 0.0000, 0.0000],
        [0.1652, 0.1680, 0.1686, 0.1633, 0.0000, 0.0000],
        [0.1655, 0.1667, 0.1672, 0.1650, 0.1747, 0.0000],
        [0.1646, 0.1690, 0.1698, 0.1615, 0.1796, 0.1554]],
       grad_fn=<MulBackward0>)

In [127]:
row_sums = masked_simple.sum(dim=-1, keepdim=True)
masked_simple_norm = masked_simple/row_sums

In [128]:
masked_simple_norm 

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4917, 0.5083, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3255, 0.3362, 0.3382, 0.0000, 0.0000, 0.0000],
        [0.2483, 0.2527, 0.2535, 0.2456, 0.0000, 0.0000],
        [0.1972, 0.1987, 0.1992, 0.1966, 0.2082, 0.0000],
        [0.1646, 0.1690, 0.1698, 0.1615, 0.1796, 0.1554]],
       grad_fn=<DivBackward0>)

In [129]:
#Instead of doing normalisation twice. mask after attention score and then normalise


In [131]:
mask = torch.triu(torch.ones(context_length,context_length),diagonal=1)
masked = attn_scores.masked_fill(mask.bool(),-torch.inf)
print(masked)

tensor([[ 0.0879,    -inf,    -inf,    -inf,    -inf,    -inf],
        [ 0.0663,  0.1132,    -inf,    -inf,    -inf,    -inf],
        [ 0.0638,  0.1095,  0.1180,    -inf,    -inf,    -inf],
        [ 0.0349,  0.0593,  0.0637,  0.0190,    -inf,    -inf],
        [ 0.0006,  0.0115,  0.0152, -0.0035,  0.0776,    -inf],
        [ 0.0618,  0.0998,  0.1059,  0.0355,  0.1857, -0.0190]],
       grad_fn=<MaskedFillBackward0>)


In [134]:
masked_attn_weights = torch.softmax(masked/ keys.shape[-1]**0.5,dim=-1)

In [135]:
masked_attn_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4917, 0.5083, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3255, 0.3362, 0.3382, 0.0000, 0.0000, 0.0000],
        [0.2483, 0.2527, 0.2535, 0.2456, 0.0000, 0.0000],
        [0.1972, 0.1987, 0.1992, 0.1966, 0.2082, 0.0000],
        [0.1646, 0.1690, 0.1698, 0.1615, 0.1796, 0.1554]],
       grad_fn=<SoftmaxBackward0>)

### Dropout mask - to reduce over fitting

In [141]:
#Dropout masks are not used by modern GPT models


In [148]:
torch.manual_seed(7293)

layer = torch.nn.Dropout(0.5)

In [149]:
example = torch.ones(6,6)

In [150]:
layer(example)

tensor([[0., 2., 0., 2., 0., 2.],
        [2., 0., 0., 0., 2., 0.],
        [0., 2., 2., 0., 2., 2.],
        [0., 2., 2., 2., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [2., 0., 2., 2., 0., 2.]])

In [151]:
layer(masked_attn_weights)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.6724, 0.6765, 0.0000, 0.0000, 0.0000],
        [0.4967, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3944, 0.0000, 0.0000, 0.0000, 0.4165, 0.0000],
        [0.0000, 0.3381, 0.0000, 0.3231, 0.0000, 0.0000]],
       grad_fn=<MulBackward0>)

### Compact Causal Self Attention Class

In [167]:
batch = torch.stack((inputs,inputs),dim=0)
batch.shape

torch.Size([2, 6, 3])

In [171]:
import torch.nn as nn

class CausalAttention(nn.Module):
    
    def __init__(self,d_in,d_out,dropout,context_length,qkv_bias = False):
        super().__init__()
        self.W_query = torch.nn.Linear(d_in,d_out,bias = qkv_bias)
        self.W_key = torch.nn.Linear(d_in,d_out,bias = qkv_bias)
        self.W_value = torch.nn.Linear(d_in,d_out,bias = qkv_bias)
        self.dropout = torch.nn.Dropout(dropout)
        self.register_buffer("mask",torch.triu(torch.ones(context_length,context_length),diagonal=1)) 
        # This is useful to run in local GPU - it tranfers the tensor to local
        

    def forward(self,x):
        b,num_tokens,d_in = x.shape        
        query = self.W_query(x)
        keys =  self.W_key(x)
        values =  self.W_value(x)

        attn_scores = query @ keys.transpose(1,2) # In batch operation dimensions x.shape = 2*6*2, shape of matrix is last two dimension
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens,:num_tokens],-torch.inf)

        #masked_fill_ new _ops are in place it directly modifies the attn_scores
        
        attn_weights = torch.softmax(masked/ keys.shape[-1]**0.5,dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vects = attn_weights @ values

        return context_vects
        
torch.manual_seed(798)
dropout = 0
context_length = batch.shape[1]
ca= CausalAttention(d_in,d_out,0.5,context_length)
#ca(batch)      
        

In [172]:
query

tensor([[0.4787, 0.4642],
        [0.4016, 0.5321],
        [0.3898, 0.5275],
        [0.2096, 0.2734],
        [0.0686, 0.2974],
        [0.3389, 0.3377]], grad_fn=<MmBackward0>)

In [173]:
keys

tensor([[ 0.2342, -0.0521],
        [ 0.3325, -0.0382],
        [ 0.3397, -0.0274],
        [ 0.1510, -0.0465],
        [ 0.3740,  0.1746],
        [ 0.0947, -0.1513]], grad_fn=<MmBackward0>)

In [174]:
keys.T

tensor([[ 0.2342,  0.3325,  0.3397,  0.1510,  0.3740,  0.0947],
        [-0.0521, -0.0382, -0.0274, -0.0465,  0.1746, -0.1513]],
       grad_fn=<PermuteBackward0>)

 ### Multihead Causal Attention

In [185]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self,d_in,d_out,dropout,context_length,num_heads=2,qkv_bias = False):
        super().__init__()
        self.heads = nn.ModuleList([
            CausalAttention(d_in,d_out,dropout,context_length,qkv_bias) for _ in range(num_heads)
        ])

    def forward(self,x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

torch.manual_seed(123)

context_length = batch.shape[1]
d_in,d_out = batch.shape[-1],2
mha = MultiHeadAttentionWrapper(d_in, d_out, 0.0, context_length, num_heads=2)
mha(batch)

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5852,  0.0093,  0.5787,  0.3054],
         [-0.6283, -0.0605,  0.6117,  0.3694],
         [-0.5669, -0.0827,  0.5472,  0.3526],
         [-0.5492, -0.0984,  0.5311,  0.3401],
         [-0.5304, -0.1073,  0.5092,  0.3431]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5852,  0.0093,  0.5787,  0.3054],
         [-0.6283, -0.0605,  0.6117,  0.3694],
         [-0.5669, -0.0827,  0.5472,  0.3526],
         [-0.5492, -0.0984,  0.5311,  0.3401],
         [-0.5304, -0.1073,  0.5092,  0.3431]]], grad_fn=<CatBackward0>)

### Multihead Causal Attention - Parallel computing

In [187]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert (d_out % num_heads == 0), \
            "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads # Reduce the projection dim to match desired output dim

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # Linear layer to combine head outputs
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length),
                       diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        # As in `CausalAttention`, for inputs where `num_tokens` exceeds `context_length`, 
        # this will result in errors in the mask creation further below. 
        # In practice, this is not a problem since the LLM (chapters 4-7) ensures that inputs  
        # do not exceed `context_length` before reaching this forward method.

        keys = self.W_key(x) # Shape: (b, num_tokens, d_out)
        queries = self.W_query(x)
        values = self.W_value(x)

        # We implicitly split the matrix by adding a `num_heads` dimension
        # Unroll last dim: (b, num_tokens, d_out) -> (b, num_tokens, num_heads, head_dim)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim) 
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        # Transpose: (b, num_tokens, num_heads, head_dim) -> (b, num_heads, num_tokens, head_dim)
        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        # Compute scaled dot-product attention (aka self-attention) with a causal mask
        attn_scores = queries @ keys.transpose(2, 3)  # Dot product for each head

        # Original mask truncated to the number of tokens and converted to boolean
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]

        # Use the mask to fill attention scores
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        # Shape: (b, num_tokens, num_heads, head_dim)
        context_vec = (attn_weights @ values).transpose(1, 2) 
        
        # Combine heads, where self.d_out = self.num_heads * self.head_dim
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec) # optional projection

        return context_vec

torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)

context_vecs = mha(batch)

print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])
